In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/road_fine.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "dismissal": "string",
        "vehicleClass": "string",
        "notificationType": "string",
        "lastSent": "string",
        "amount": "float32",
        "totalPaymentAmount": "float32",
        "article": "float32",
        "points": "float32",
        "expense": "float32",
        "paymentAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,amount,article,concept:name,dismissal,expense,lastSent,lifecycle:transition,notificationType,org:resource,paymentAmount,points,time_delta,totalPaymentAmount,vehicleClass
0,A1,2006-07-24,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
1,A1,2006-12-05,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11577600.0,0.0,NA
2,A100,2006-08-02,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
3,A100,2006-12-12,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11404800.0,0.0,NA
4,A100,2007-01-15,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,2937600.0,0.0,NA
5,A100,2007-03-16,71.5,0.0,Add penalty,NA,0.0,NA,complete,NA,NA,0.0,0.0,5184000.0,0.0,NA
6,A100,2009-03-30,0.0,0.0,Send for Credit Collection,NA,0.0,NA,complete,NA,NA,0.0,0.0,64368000.0,0.0,NA
7,A10000,2007-03-09,36.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
8,A10000,2007-07-17,0.0,0.0,Send Fine,NA,13.0,NA,complete,NA,NA,0.0,0.0,11232000.0,0.0,NA
9,A10000,2007-08-02,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,1382400.0,0.0,NA


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['amount', 'article', 'concept:name', 'dismissal', 'expense', 'lastSent', 'lifecycle:transition', 'notificationType', 'org:resource', 'paymentAmount', 'points', 'time_delta', 'totalPaymentAmount', 'vehicleClass']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 8726400.00]                       2160000.0000 quantile_derived    
amount                         continuous     event    yes    [24.00, 80.00]                           6.7000     quantile_derived    
totalPaymentAmount             continuous     event

In [7]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [8]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [9]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [10]:
scenario_df.head()

,case:concept:name,time_index,fake,amount,article,concept:name,dismissal,expense,lastSent,lifecycle:transition,notificationType,org:resource,paymentAmount,points,time_delta,totalPaymentAmount,vehicleClass
0,f_135630,0,True,74.838999,38.839909,Create Fine,$,9.507318,P,complete,P,838,66.295217,0.0,5.439624e+06,33.433974,C
1,f_135630,1,True,75.120715,141.158814,Insert Date Appeal to Prefecture,4,11.106577,N,complete,P,51,32.982632,0.0,8.070304e+05,30.155390,A
2,f_135630,2,True,41.873323,22.733354,Send for Credit Collection,R,11.369354,C,complete,P,61,71.723702,0.0,7.609102e+06,6.168820,R
3,f_135630,3,True,58.570403,101.133921,Payment,5,16.473580,P,complete,C,813,50.527598,0.0,8.256768e+06,26.678823,M
4,f_135630,4,True,64.770352,50.685705,Notify Result Appeal to Offender,@,9.399322,N,complete,P,44,47.213449,0.0,6.607400e+06,11.998002,A


In [11]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [12]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [13]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [15]:
criterion = torch.nn.BCEWithLogitsLoss()

In [16]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/road_fine-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0000 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0000 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0000 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 44210.302222 seconds
Time taken for scenario model (validation): 34.237418 seconds
Val loss: {'loss': 0.0, 'accuracy': 1.0, 'f1_macro': 1.0, 'f1_weighted': 1.0}


In [17]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [18]:
# scenario_model = ScenarioLSTM.load()

In [19]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

### --- Cleanup ---

In [20]:
sys.stdout = original_stdout
log_file.close()